# 03 Paired Family Structure

Once the thesis established that hybrid and paired domains are far apart, the next question became: is the paired target domain itself homogeneous? The answer is no, and this notebook explains why that matters.

**Questions answered here**
- How different are the paired families from one another?
- What happens when one family is fully unseen during training?
- Why do some families transfer well while others break?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("03_paired_family_structure")
domain_bundle = get_domain_shift_bundle()
limitations = get_limitations_bundle()
family_coverage = build_family_coverage_table()


## 1. Family coverage inside the paired domain

The paired benchmark is not just small; it is uneven. Some families contribute many more units or recordings than others, which already creates a robustness challenge before any model is trained.


In [ ]:
display(family_coverage)
save_table(family_coverage, table_dir, "paired_family_coverage")


## 2. Geometry of the paired families

The paired-only PCA shows whether families occupy distinct regions or overlap cleanly. The important point is not perfect separation, but visible heterogeneity.


In [ ]:
paired_only = domain_bundle.pca_2d[domain_bundle.pca_2d["dataset_type"] == "paired"].copy()
save_table(paired_only, table_dir, "paired_pca_points")
fig, _ = plot_pca_projection(paired_only, color_col="study_set")
save_figure(fig, fig_dir, "paired_family_pca_2d")
fig


## 3. Leave-one-family-out stress test

The main benchmark is still recording-disjoint. But this stress test asks a harsher question: what happens when an entire paired family is unseen? The answer shows which families behave like bridge domains and which behave like island domains.


In [ ]:
display(limitations.lofo_aggregate)
display(limitations.lofo_family)
save_table(limitations.lofo_aggregate, table_dir, "leave_one_family_aggregate")
save_table(limitations.lofo_family, table_dir, "leave_one_family_by_family")
fig, _, _ = plot_group_metric(limitations.lofo_family, group_col="held_out_family", metric="r2", title="Leave-one-family-out `fpos` R² by held-out family")
save_figure(fig, fig_dir, "leave_one_family_r2_by_family")
fig


In [ ]:
show_saved_figure(
    fig_dir / "recording_vs_family_disjoint_r2_gap.png",
    "This summary compares the best recording-disjoint `fpos` waveform-augmented stack with its leave-one-family-out performance, which is the cleanest way to show that family-unseen generalization is a stricter transfer problem than the main benchmark.",
)


## 4. Why some families improve and others crash

The interesting behavior in the leave-one-family-out benchmark is that some held-out families remain strong while others collapse. That happens because each held-out family defines a different transfer problem: some are reconstructible from the remaining families, others are not.


In [ ]:
pivot = limitations.lofo_family.pivot_table(index="held_out_family", columns="variant_id", values="r2")
pivot["waveform_minus_reference"] = pivot["wf_embed_anchor_stack_xgboost"] - pivot["reference_anchor_stack_xgboost"]
gap_table = pivot.reset_index().sort_values("waveform_minus_reference")
display(gap_table)
save_table(gap_table, table_dir, "leave_one_family_gap_table")


In [ ]:
display(Markdown(
    """
## Key takeaways

- The paired domain is not one clean target domain; it is a mixture of families with different transfer behavior.
- Leave-one-family-out is therefore a real robustness test, not just a stricter version of the main benchmark.
- The waveform-augmented `fpos` stack is not uniformly best under unseen-family transfer, which is why the thesis reports both optimization and robustness protocols.
"""
))


---
### Thesis highlights — Family structure preference map and protocol tradeoff

_Provenance._ These are thesis synthesis figures regenerated from frozen tables rather than notebook-local plotting code.

- **Family structure preference map:** The left panel keeps the family-by-model preference heatmap. The right panel adds actual family structure variables from the local package, so the figure now shows not only _which_ family prefers which recipe, but also _how different regimes sit relative to one another_.
- **Protocol tradeoff:** This figure is sourced directly from the curated `fpos_protocol_regime_preference.csv` table, so the notebook is displaying the same protocol comparison that the thesis cites.


In [ ]:
show_saved_figure(
    ROOT / "figures" / "09_thesis_highlight_figures" / "family_structure_preference_map.png",
    "This synthesis figure combines family-level performance preference with structural context from the local package.",
)
show_saved_figure(
    ROOT / "figures" / "09_thesis_highlight_figures" / "protocol_tradeoff.png",
    "The protocol figure is sourced from the frozen curated comparison table rather than reconstructed ad hoc inside the notebook.",
)
